In [ ]:
# 0) Colab: mount Drive (train-data + runs) + clone code từ GitHub
# Chạy được bằng BẤT KỲ account nào (chính HOẶC phụ) — dùng CHUNG 1 path, không sửa code khi đổi account.
#
# YÊU CẦU 1 LẦN cho account phụ (account chính bỏ qua):
#   Mở link share (quyền Editor) -> chuột phải folder "recommender_train_colab"
#   -> Organize/Sắp xếp -> Add shortcut to Drive (Thêm lối tắt) -> My Drive.
#   Shortcut giữ nguyên tên => path /content/drive/MyDrive/recommender_train_colab
#   giống hệt account chính. Editor => ghi checkpoint/runs.csv thẳng vào folder gốc.
#
# ⚠ TRAIN-DATA V2: Drive phải chứa train-data ĐÃ REBUILD (cold_items + eval_seen +
#   examples {val,test}_cold + history full). Bản v1 KHÔNG chạy được với code v2.
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

DRIVE_DIR = '/content/drive/MyDrive/recommender_train_colab'
assert os.path.isdir(DRIVE_DIR), (
    f"Khong thay {DRIVE_DIR}.\n"
    "Account phu: mo link share (Editor) trong Drive roi 'Add shortcut to My Drive' "
    "cho folder recommender_train_colab. Account chinh: kiem tra folder con o My Drive."
)

REPO = 'https://github.com/CrocodixD/anime-recommender.git'
CODE = '/content/recommender'

if os.path.exists(CODE):
    !cd {CODE} && git pull
else:
    !git clone {REPO} {CODE}

# train-data nằm ở retriever/train-data (ROOT của src/config.py = retriever/)
if not os.path.exists(f'{CODE}/retriever/train-data'):
    !cp -r "{DRIVE_DIR}/train-data" "{CODE}/retriever/train-data"

%cd {CODE}

In [ ]:
# 1) Imports + path (code train ở retriever/src — import flat)
# imp shim: vài lib trên Colab (py3.12) còn import 'imp' đã bị gỡ khỏi stdlib
import sys, importlib
sys.modules['imp'] = importlib

import pathlib
HERE = pathlib.Path.cwd()
SRC_DIR = HERE / 'retriever' / 'src' if (HERE / 'retriever' / 'src').exists() else HERE
sys.path.insert(0, str(SRC_DIR))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import config, data, model as M, loss, metrics, train
print('torch', torch.__version__)

In [ ]:
# 2) Đường dẫn persistence trên Drive (checkpoint + history + leaderboard)
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/recommender_train_colab')
VERSION = 'v5'                                # v5 = data v2 + protocol v2 (KHÔNG so số với v1-v4)
RUNS_DIR = DRIVE / 'runs' / VERSION            # mỗi run: runs/<VERSION>/<run_name>/best.pt + history.json
RESULTS_CSV = DRIVE / 'runs.csv'               # leaderboard CHUNG mọi version (có cột 'version')
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print('version', VERSION, '· runs ->', RUNS_DIR)

In [ ]:
# 3) Base config T4 (Colab free: 2 core / 13GB RAM / 15GB VRAM). ckpt_dir set per-run ở helper (cell 5).
# PROTOCOL V2: headline = warm recall@200 (val); eval_ks tới 500; mask seen−query.
# Mọi knob mới có flag bật/tắt (search lật được) — xem retriever/CLAUDE.md §Config surface.
base_cfg = config.TwoTowerConfig(
    # model
    d=128,
    history_source='cache',      # 'embed' = bảng history trainable (đòn bẩy chính cần thử)

    # synopsis (content text-emb item-side; CẦN synopsis_emb.npy + synopsis_low_info.npy trong train-data)
    use_synopsis=False,          # True = bật nhánh synopsis (sinh local bằng 07 rồi đẩy 2 .npy lên Drive train-data/)
    synopsis_dim=48,             # chiều sau chiếu; synopsis_proj_hidden=[] (Linear thuần) | [128] nếu underfit
    synopsis_normalize='none',   # artifact đã L2 sẵn ('l2'/'standardize' nếu đổi cách chuẩn hoá)

    # loss
    tau=0.07,
    beta=1.0,
    logq_alpha=1.0,              # hệ số logQ (thử 0.5/0.75 — metric thưởng popularity)

    # train
    lr=1e-3,
    optimizer='adam',           # 'adamw' = decouple weight_decay (đòn bẩy regularization khi search)
    weight_decay=0.0,
    batch_size=16384,
    epochs=1,
    hist_dropout=0.12,
    m_hardneg=5,
    train_hist_len=32,           # sample history/anchor từ FULL list (augmentation)
    max_examples_per_user=None,  # vd 64: epoch ngắn ~4x + cân trọng số per-user
    # train_user_frac=0.15,      # HP-search COARSE: chỉ giữ % user (full catalog+logQ+eval không đổi); None = full

    # eval (protocol v2)
    eval_every_steps=500,

    # infra
    num_workers=2,
    # subset=200_000,            # bỏ comment để chạy thử nhanh; None = full
)
print('device', base_cfg.device)   # 'cuda' trên T4
base_cfg

In [ ]:
# 4) Build module (sanity-check shape / param count)
spec, logq, item_table, users, mdl = train.build(base_cfg)
n_params = sum(p.numel() for p in mdl.parameters() if p.requires_grad)
print(f'num_items={item_table.num_items}  num_users={users.num_users}  params={n_params:,}')
print('logq', logq.shape, 'finite', int(torch.isfinite(logq).sum()))

In [ ]:
# 5) Helpers thử nghiệm: mỗi run có tên riêng -> best.pt/history.json/row.json/config.json.
# GHI NÓNG ra disk LOCAL (/content) lúc train -> tránh race Drive FUSE (folder mới chưa settle +
# best.pt ghi lại nhiều lần). Cuối mỗi run SYNC sang Drive 1 LẦN (copytree). Leaderboard (runs.csv)
# vẫn là artifact SUY RA từ mọi best.pt trên Drive (rebuild_leaderboard) -> bền với run ngắt + nhiều account.
import json, time, dataclasses, shutil

LOCAL_RUNS = Path('/content/runs_local')               # disk ephemeral của Colab (mất khi hết session, không sao)

def _cfg_from_ckpt(saved_cfg):
    """cfg trong ckpt mang path/device của session train CŨ + có thể THIẾU field mới
    (ckpt v1-v4 pickle theo class cũ). Fill default cho field thiếu, override path/device
    về môi trường hiện tại -> build/eval lại không lỗi."""
    base = config.TwoTowerConfig()
    kw = {f.name: getattr(saved_cfg, f.name, getattr(base, f.name))
          for f in dataclasses.fields(config.TwoTowerConfig)}
    kw['device'] = config.auto_device()
    kw['train_data'] = config.TRAIN_DATA
    return config.TwoTowerConfig(**kw)

def load_run_dir(run_dir):
    """Load best.pt từ 1 thư mục bất kỳ (local hoặc Drive) -> (cfg, ckpt, model, users, logq).
    Dựng model theo cfg LƯU TRONG checkpoint (gồm mọi override d/tau/...) -> khớp shape state_dict;
    chỉ thay path/device về môi trường hiện tại (xem _cfg_from_ckpt)."""
    ckpt = torch.load(Path(run_dir) / 'best.pt', weights_only=False)  # file kèm cfg -> cần False
    cfg = _cfg_from_ckpt(ckpt['cfg'])
    _, logq, _, users, model = train.build(cfg)
    model.load_state_dict(ckpt['model'])
    model.refresh_item_cache()
    return cfg, ckpt, model, users, logq

def load_run(run_name):
    """Load best.pt của 1 run từ Drive (RUNS_DIR/<run_name>) -> sẵn sàng eval/inference."""
    return load_run_dir(RUNS_DIR / run_name)

def eval_run(cfg, model, users, logq):
    """Eval WARM val + test (protocol v2: mask seen−query) -> dict phẳng cho leaderboard.
    Cold slice KHÔNG vào leaderboard (final-exam discipline) — dùng cell 10."""
    out = {}
    seen = data.load_eval_seen(cfg.train_data)
    for split in ['val', 'test']:
        ds = data.ExamplesDataset(cfg.train_data, split)
        q = metrics.group_examples(ds.user_idx, ds.anime_idx)
        masks = metrics.build_masks(seen, q)
        m = metrics.evaluate(model, users, q, logq, cfg.eval_ks, masks,
                             eval_history_cap=cfg.eval_history_cap)
        for k in cfg.eval_ks:
            out[f'{split}_recall@{k}'] = round(float(m[f'recall@{k}']), 4)
            out[f'{split}_ndcg@{k}']   = round(float(m[f'ndcg@{k}']), 4)
    return out

def _make_row(run_name, version, cfg, ckpt, model, users, logq, secs=None, ts=None):
    """1 dòng leaderboard từ (cfg, ckpt, model đã load) — eval lại val+test. Dùng chung run mới + khôi phục."""
    return {
        'run_name': run_name, 'version': version, 'time': ts,
        'd': cfg.d, 'tau': cfg.tau, 'beta': cfg.beta, 'lr': cfg.lr,
        'cosine_lr': cfg.cosine_lr, 'weight_decay': cfg.weight_decay, 'optimizer': cfg.optimizer,
        'batch_size': cfg.batch_size, 'epochs': cfg.epochs,
        'hist_dropout': cfg.hist_dropout, 'm_hardneg': cfg.m_hardneg,
        'use_item_id': cfg.use_item_id, 'id_dim': cfg.id_dim, 'id_dropout': cfg.id_dropout,
        'score_pool': cfg.score_pool,
        'history_pool': cfg.history_pool,
        'history_source': cfg.history_source, 'train_hist_len': cfg.train_hist_len,
        'logq_alpha': cfg.logq_alpha, 'max_examples_per_user': cfg.max_examples_per_user,
        'eval_history_cap': cfg.eval_history_cap,
        'use_synopsis': cfg.use_synopsis, 'synopsis_dim': cfg.synopsis_dim,
        'synopsis_normalize': cfg.synopsis_normalize,
        'train_user_frac': cfg.train_user_frac, 'subset_seed': cfg.subset_seed,
        'best_step': ckpt.get('step'), 'secs': secs,
        **eval_run(cfg, model, users, logq),
    }

def rebuild_leaderboard():
    """runs.csv = gom MỌI run có best.pt (mọi version). Nguồn 1 row, ưu tiên:
    row.json (cache durable) > dòng cũ trong runs.csv (migrate sang row.json) >
    eval lại từ best.pt (run bị ngắt giữa chừng -> dựng row, cache lại). KHÔNG train lại.
    Dedupe theo (version, run_name). Lưu ý: row v1-v4 đo theo protocol CŨ — đừng so
    thẳng với v5 (cột test_recall@200 chỉ có từ v5)."""
    old = {}
    if RESULTS_CSV.exists():
        for r in pd.read_csv(RESULTS_CSV).to_dict('records'):
            old[(str(r.get('version')), str(r.get('run_name')))] = r
    rows, seen = [], set()
    for ckpt_path in sorted((DRIVE / 'runs').glob('*/*/best.pt')):
        run_dir = ckpt_path.parent
        run_name, version = run_dir.name, run_dir.parent.name
        if (version, run_name) in seen:                                 # folder trùng tên -> bỏ qua
            continue
        seen.add((version, run_name))
        row_file = run_dir / 'row.json'
        if row_file.exists():
            row = json.loads(row_file.read_text())
        elif (version, run_name) in old:
            row = old[(version, run_name)]
            row_file.write_text(json.dumps(row, default=float))         # migrate dòng cũ -> durable
        else:                                                           # run bị ngắt -> eval lại từ best.pt
            cfg, ckpt, model, users, logq = load_run_dir(run_dir)
            row = _make_row(run_name, version, cfg, ckpt, model, users, logq)
            row_file.write_text(json.dumps(row, default=float))
            print(f"  recovered {version}/{run_name} (eval từ best.pt)")
        rows.append(row)
    df = pd.DataFrame(rows)
    df.to_csv(RESULTS_CSV, index=False)
    col = 'test_recall@200' if 'test_recall@200' in df.columns else 'test_recall@100'
    return df.sort_values(col, ascending=False, na_position='last')

def run_experiment(run_name, **overrides):
    """Train ra DISK LOCAL (/content/runs_local) -> tránh race Drive FUSE; cuối run SYNC nguyên
    thư mục sang Drive 1 LẦN (copytree). overrides là field của TwoTowerConfig
    (vd use_item_id=True, id_dim=128, history_source='embed', use_synopsis=True, train_user_frac=0.15)."""
    cfg = dataclasses.replace(base_cfg, **overrides)
    local_dir = LOCAL_RUNS / VERSION / run_name
    local_dir.mkdir(parents=True, exist_ok=True)
    cfg.ckpt_dir = local_dir                            # fit ghi best.pt vào LOCAL (nhanh, không flaky)
    t0 = time.time()
    model, history = train.fit(cfg)
    secs = time.time() - t0
    (local_dir / 'history.json').write_text(json.dumps(history, default=float))
    (local_dir / 'config.json').write_text(json.dumps(dataclasses.asdict(cfg), default=str))  # full cfg = provenance đồ án
    # eval lại từ best.pt LOCAL (model tốt nhất, không phải step cuối) -> row.json
    cfg, ckpt, model_e, users_e, logq_e = load_run_dir(local_dir)
    row = _make_row(run_name, VERSION, cfg, ckpt, model_e, users_e, logq_e,
                    secs=round(secs), ts=pd.Timestamp.now().strftime('%Y-%m-%d %H:%M'))
    (local_dir / 'row.json').write_text(json.dumps(row, default=float))
    # SYNC sang Drive 1 lần: parent RUNS_DIR đã settle từ cell 2 -> tạo 1 folder con (không race)
    RUNS_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local_dir, RUNS_DIR / run_name, dirs_exist_ok=True)
    print(f"\n✓ {run_name}: test recall@200={row['test_recall@200']}  (synced -> {RUNS_DIR / run_name})")
    rebuild_leaderboard()                              # runs.csv = gom mọi run (durable, tự lành)
    return cfg, history, row

In [ ]:
# 6) TRAIN BẢN FINAL — config CHỐT từ warm (runs.csv) + cold (cold_runs.csv) leaderboard, namespace 'v_final'.
#    CHỐT: embed · train_hist_len=128 · history_pool=mean · score_pool=none · d=128 · id_dim=128 id_dropout=0.15 ·
#          tau=0.07 · logq_alpha=1 · adam · bs=16384 · KHÔNG synopsis · KHÔNG cosine/lr-bump.
#    Bằng chứng: hl128 thắng warm recall@200 (.6777) + cold recall@200 (.4359) + cold ndcg@10 (.1694);
#                attn & (cosine+lr1.5e-3+ep8) đều HẠI cold; synopsis KHÔNG giúp cold (cold_runs.csv).
#    Khác lần confirm: epochs=8 + EARLY STOP theo recall@200 val -> TỰ DỪNG khi hội tụ. Lần trước phải dừng TAY
#    vì early_stop_patience=None (=TẮT). Giờ patience=6, min_delta=0.0: dừng khi KHÔNG có peak recall@200 mới
#    suốt 6 eval (plateau thật); KHÔNG cắt lúc còn leo (mỗi eval lập peak mới -> reset). [min_delta=0.001 SAI khi
#    eval mỗi 500 step: gain/eval lúc leo thật <0.001 -> cắt sớm ~epoch 0.9, mất ~.02 recall — đã mô phỏng.]
#    best.pt = checkpoint TỐT NHẤT theo recall@200 (không phải step cuối); epochs=8 = trần chặn (luôn tự kết thúc).
#    ⚠ bs=16384 như lần confirm (cần A100). Colab T4 OOM -> hạ batch_size=8192 (ít in-batch neg hơn, chấp nhận được).
def run_final(run_name, **overrides):
    """Như run_experiment (cell 5) nhưng SYNC sang runs/v_final — tách bản final khỏi v5 experiments.
    Swap VERSION/RUNS_DIR trong lúc gọi rồi KHÔI PHỤC (finally, kể cả lỗi) -> cell 6b/9/10 chạy sau vẫn thấy v5."""
    global VERSION, RUNS_DIR
    _v, _r = VERSION, RUNS_DIR
    VERSION, RUNS_DIR = 'v_final', DRIVE / 'runs' / 'v_final'
    RUNS_DIR.mkdir(parents=True, exist_ok=True)   # tạo sớm (như cell 2 cho v5) -> Drive FUSE settle trước copytree cuối run
    try:
        return run_experiment(run_name, **overrides)
    finally:
        VERSION, RUNS_DIR = _v, _r

FINAL = dict(use_item_id=True, id_dim=128, id_dropout=0.15, history_source='embed', train_hist_len=128,
             epochs=8, early_stop_patience=6, early_stop_min_delta=0.0,
             eval_cold_in_loop=True)   # cold val theo step -> history; mọi knob khác = base_cfg (cell 3)

# (1) BẢN FINAL (không synopsis) — model này sẽ export -> artifacts -> ranker.
cfg, history, row = run_final('final', **FINAL)

# (2) TWIN + synopsis (dim=64 = cấu hình syn tốt nhất ở các run trước) — CÙNG config, chỉ khác use_synopsis
#     -> cặp ablation SẠCH (full data) để CHỐT "synopsis không giúp", chấm cold ở cell 10.
#     Cần synopsis_emb.npy + synopsis_low_info.npy trong train-data trên Drive. Comment nếu chỉ muốn bản final.
cfg_s, history_s, row_s = run_final('final_syn', use_synopsis=True, synopsis_dim=64,
                                    synopsis_normalize='none', **FINAL)


In [ ]:
# 6b) (TUỲ CHỌN) SWEEP SYNOPSIS trên SUBSET 15% — bằng chứng BỔ SUNG cho cặp full-data ở cell 6:
#     kể cả quét synopsis_dim + chiều sâu projection ở config TỐI ƯU (embed/hl128/mean), synopsis vẫn KHÔNG
#     vượt control no-syn trên cold (chấm bằng cell 10). Cặp cell 6 là bằng chứng CHÍNH; cell này thêm bề rộng.
#     [Search KIẾN TRÚC attn/score_pool/tau/mhn đã XONG & loại: attn HẠI cold, mhn 5≈10 — cold_runs.csv.]
#     KHÔNG quét synopsis_normalize: artifact đã L2 sẵn (07) -> 'l2' = no-op (F.normalize vec đơn vị),
#     'standardize' CHƯA implement trong model.py (rơi về none) => normalize là knob chết. Thay bằng proj_hidden
#     (Linear vs 1-hidden) = đòn bẩy THẬT, test synopsis có underfit không (EXPERIMENTS §1).
import search, importlib; importlib.reload(search)

space = {
    'use_synopsis':         [False, True],   # False = CONTROL: canonicalize bỏ con -> 1 config no-syn cùng base
    'synopsis_dim':         [48, 64],        # chiều nhánh synopsis sau khi chiếu
    'synopsis_proj_hidden': [[], [128]],     # [] = Linear thuần | [128] = 1 hidden (test underfit)
}
# base TỐI ƯU (khớp cell 6) — chỉ khác: subset 15% user + ep2 cho rẻ. mhn=5/tau=.07/mean/spool=none/snorm=none từ base_cfg.
fixed = {'use_item_id': True, 'id_dim': 128, 'id_dropout': 0.15, 'history_source': 'embed',
         'train_hist_len': 128, 'epochs': 2, 'train_user_frac': 0.15}
exists = lambda name: (RUNS_DIR / name).exists()       # resume: skip run đã có trên Drive (runs/v5)

# grid = vét cạn -> 5 config (1 control no-syn + 4 synopsis: dim×proj_hidden). dry_run=True: chỉ in tên (KHÔNG train).
# Bỏ dry_run để train thật (-> v5), rồi chạy cell 10 so cold các dòng syn vs control (cùng base).
sweep = search.iter_configs(space, method='grid', fixed=fixed)
results = search.run_search(run_experiment, sweep, exists_fn=exists, dry_run=True)


In [ ]:
# 6c) (TUỲ CHỌN) ABLATION LOSS — one-factor-at-a-time trên SUBSET 15% (coarse rank).
#   Bằng chứng "từng thành phần loss có ích không": mỗi knob 1 grid riêng (KHÔNG cross-product),
#   fixed = config final + subset/ep2 cho rẻ. Run reference (mọi knob=default) lặp giữa các sweep
#   -> deterministic_run_name + exists_fn tự dedupe (không train lại). So kết quả: cell 9 (warm runs.csv)
#   + cell 11 (cold cold_runs.csv) đã gom mọi best.pt. CONFIRM phát hiện đầu bảng trên FULL bằng cell 6.
#   Bỏ dry_run=True để train thật (-> v5). m_hardneg=0 = tắt nhánh hard-neg; logq_alpha=0 = tắt logQ.
import search, importlib; importlib.reload(search)

fixed = dict(use_item_id=True, id_dim=128, id_dropout=0.15, history_source='embed',
             train_hist_len=128, epochs=2, train_user_frac=0.15)   # khớp config final + coarse
exists = lambda name: (RUNS_DIR / name).exists()                   # resume: skip run đã có (runs/v5)
loss_space = [
    {'m_hardneg':   [0, 5, 10]},        # hard-neg OFF / 5 / 10
    {'logq_alpha':  [0.0, 0.5, 1.0]},   # logQ OFF / half / full
    {'tau':         [0.05, 0.07, 0.1]}, # temperature (mean-pool, sạch — khác attn runs cũ)
    {'beta':        [0.5, 1.0, 2.0]},   # trọng số hard-neg trong mẫu số (chưa từng != 1)
    {'id_dropout':  [0.15, 0.3, 0.5]},  # đòn bẩy TRỰC TIẾP lên gap cold (overfit-vào-id)
]
for sp in loss_space:
    search.run_search(run_experiment, search.iter_configs(sp, method='grid', fixed=fixed),
                      exists_fn=exists, dry_run=True)   # dry_run=True: chỉ in tên; bỏ để train thật


In [ ]:
# 9) Leaderboard: gom MỌI run có best.pt (kể cả run bị ngắt giữa chừng -> eval lại từ best.pt).
# Headline retriever v5 = test_recall@200 (warm). Row v1-v4 đo protocol cũ — đừng so thẳng.
rebuild_leaderboard()

In [ ]:
# 10) COLD LEADERBOARD — eval cold VAL mọi run (v5 + v_final) -> Drive/cold_runs.csv (song song runs.csv).
# Đo khả năng gợi ý anime MỚI: H encode id->OOV (content thật) qua run_cold_eval -> refresh_item_cache(cold_mask).
# DISCIPLINE: leaderboard CHỈ dùng split='val' (val_cold); test_cold = FINAL EXAM, chấm 1 lần lúc chốt (KHÔNG ở đây).
# Durable như runs.csv: cache cold_row.json/run -> chạy lại chỉ eval run MỚI (FORCE=True để tính lại hết).
# Mỗi run 2 chế độ: full-catalog (recall/ndcg/hit@K) + h_only (candidate chỉ tập H — diagnostic content, recall/hit@K).
# Gom MỌI version có best.pt (cold protocol = data v2). Bar (val_cold, RESULTS §3): v5_hist64_ep2 r@200=.3881 hit@500=.5881.
COLD_SPLIT = 'val'                              # 'test' DÀNH cho final-exam — KHÔNG đổi ở leaderboard
COLD_RESULTS_CSV = DRIVE / 'cold_runs.csv'      # song song RESULTS_CSV = DRIVE/runs.csv (cell 2)
FORCE = False                                   # True = bỏ qua cache cold_row.json, eval lại mọi run

def cold_row_for(run_dir, run_name, version):
    """Eval cold (full + h_only) cho 1 run -> 1 dòng leaderboard. load_run_dir (cell 5) dựng model theo
    cfg LƯU TRONG ckpt (khớp d/synopsis/...); run_cold_eval refresh cache cold (H->OOV) trước khi chấm."""
    cfg, ckpt, model, users, logq = load_run_dir(run_dir)
    full  = metrics.run_cold_eval(model, users, cfg.train_data, logq, cfg.eval_ks, split=COLD_SPLIT, h_only=False)
    honly = metrics.run_cold_eval(model, users, cfg.train_data, logq, cfg.eval_ks, split=COLD_SPLIT, h_only=True)
    if torch.cuda.is_available():
        del model; torch.cuda.empty_cache()     # giải phóng VRAM giữa các run (loop 30+ ckpt trên T4)
    row = {
        'run_name': run_name, 'version': version, 'cold_split': COLD_SPLIT,
        'history_source': cfg.history_source, 'train_hist_len': cfg.train_hist_len,
        'use_synopsis': cfg.use_synopsis, 'synopsis_dim': cfg.synopsis_dim, 'synopsis_normalize': cfg.synopsis_normalize,
        'm_hardneg': cfg.m_hardneg, 'd': cfg.d, 'id_dim': cfg.id_dim, 'id_dropout': cfg.id_dropout,
        'tau': cfg.tau, 'logq_alpha': cfg.logq_alpha, 'optimizer': cfg.optimizer,
        'epochs': cfg.epochs, 'batch_size': cfg.batch_size, 'train_user_frac': cfg.train_user_frac,
        'best_step': ckpt.get('step'),
        'cold_n_users': full['n_users'], 'cold_n_pairs': full['n_pairs'],
    }
    for k in cfg.eval_ks:
        row[f'cold_recall@{k}']       = round(float(full[f'recall@{k}']), 4)
        row[f'cold_ndcg@{k}']         = round(float(full[f'ndcg@{k}']), 4)
        row[f'cold_hit@{k}']          = round(float(full[f'hitrate@{k}']), 4)
        row[f'cold_honly_recall@{k}'] = round(float(honly[f'recall@{k}']), 4)
        row[f'cold_honly_hit@{k}']    = round(float(honly[f'hitrate@{k}']), 4)
    return row

def rebuild_cold_leaderboard():
    """cold_runs.csv = gom MỌI run có best.pt (mọi version), eval cold VAL. Cache cold_row.json/run (durable,
    tự lành như rebuild_leaderboard) -> bền với run ngắt + nhiều account. KHÔNG train lại. 1 run lỗi -> skip."""
    rows, seen = [], set()
    for ckpt_path in sorted((DRIVE / 'runs').glob('*/*/best.pt')):   # mọi version (v5 + v_final)
        run_dir = ckpt_path.parent
        run_name, version = run_dir.name, run_dir.parent.name
        if (version, run_name) in seen:
            continue
        seen.add((version, run_name))
        cold_file = run_dir / 'cold_row.json'
        if cold_file.exists() and not FORCE:
            rows.append(json.loads(cold_file.read_text()))     # durable cache -> không eval lại
            continue
        try:
            row = cold_row_for(run_dir, run_name, version)
        except Exception as e:
            print(f"  ⚠ skip {version}/{run_name}: {type(e).__name__}: {e}")
            continue
        cold_file.write_text(json.dumps(row, default=float))
        print(f"  cold {version}/{run_name}: r@100={row['cold_recall@100']} r@200={row['cold_recall@200']} ndcg@10={row['cold_ndcg@10']}")
        rows.append(row)
    df = pd.DataFrame(rows)
    df.to_csv(COLD_RESULTS_CSV, index=False)
    print(f"\n✓ cold_runs.csv: {len(df)} run -> {COLD_RESULTS_CSV}")
    col = 'cold_recall@200' if 'cold_recall@200' in df.columns else 'cold_recall@100'
    return df.sort_values(col, ascending=False, na_position='last')

cold_lb = rebuild_cold_leaderboard()
_show = ['run_name', 'version', 'history_source', 'train_hist_len', 'use_synopsis', 'synopsis_dim',
         'cold_n_users', 'cold_recall@100', 'cold_recall@200', 'cold_ndcg@10', 'cold_hit@200', 'cold_honly_recall@200']
cold_lb[[c for c in _show if c in cold_lb.columns]]
